# Module 2 - Timeseries Analysis

This notebook performs the Rhein-Ill timeseries analysis required by `Module2_Lab.pdf`. The raw 10-minute and 15-minute observations are aggregated to monthly mean values before the statistical analysis, as required by the assignment.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

from src.data_loading import ensure_project_directories, load_project_monthly_data, save_monthly_tables
from src.section1_timeseries_review import format_timeseries_review, normalized_series_collection, run_timeseries_review
from src.section2_timeseries_modelling import analyse_acf_pacf_collection, format_order_selection
from src.section3_model_evaluation import evaluate_model_collection, format_model_evaluation
from src.section4_sediment_influence import format_sediment_influence, run_sediment_influence_analysis
from src.section5_dependency_analysis import format_dependency_results, run_dependency_analysis
from src.plotting import (
    plot_acf_pacf_grid,
    plot_dependency_scatter,
    plot_model_diagnostics,
    plot_monthly_timeseries,
    plot_sediment_yields,
    plot_synthetic_series,
)

ensure_project_directories(PROJECT_ROOT)
monthly_data = load_project_monthly_data(RAW_DATA_DIR)
save_monthly_tables(monthly_data, PROCESSED_DATA_DIR)

for station, variables in monthly_data.items():
    for variable, series in variables.items():
        print(f"{station} {variable}: {len(series)} monthly mean values from {series.index.min().date()} to {series.index.max().date()}")

## Section 1: Timeseries review

In [ ]:
# MAIN
# Run trend tests, ADF stationarity checks, and mean/trend removal for all four monthly series.
review_results = run_timeseries_review(monthly_data, alpha=0.05)
normalized_series = normalized_series_collection(review_results)

In [ ]:
# PLOT
fig_section1 = plot_monthly_timeseries(
    monthly_data,
    review_results,
    FIGURE_DIR / "section1_monthly_timeseries.png",
)
plt.show()

In [ ]:
# PRINT
print(format_timeseries_review(review_results))

### Exercise 1: Timeseries Review

#### Main Results
The analysis is based on monthly mean values, not on the original 10-minute or 15-minute data. For the two discharge series, the fitted linear trends are not significant at the 5% level: Gisingen_Q has p = 0.3952 and Diepoldsau_Q has p = 0.6136. Both Q series also pass the ADF stationarity test, so I treated them as stable enough after subtracting their mean.

The concentration series behave differently. Gisingen_C and Diepoldsau_C both have significant negative trends, with p = 0.00763 and p = 3.25e-05. Their ADF results do not clearly show stationarity, so we removed the significant linear trend instead of only subtracting the mean. After this correction, the normalized series have mean close to zero and finite variance, which is what we need before fitting AR and ARMA models.

#### Comment on Discharge Trends
Based on this result, we would not call the discharge trend a clear measurement error or a clear physical change. The Q trends are not statistically significant, so the evidence is weak. Some visual changes could still come from climate variability, regulation, catchment changes, or rating-curve updates, but this monthly dataset does not give enough support for a strong conclusion about discharge trend.

## Section 2: Timeseries Modelling

In [ ]:
# MAIN
# Compute ACF/PACF and choose simple candidate AR/ARMA orders.
# max_order=6 keeps the models small enough to explain and compare.
order_results = analyse_acf_pacf_collection(
    normalized_series,
    nlags=24,
    max_order=6,
)

In [ ]:
# PLOT
fig_section2 = plot_acf_pacf_grid(
    order_results,
    FIGURE_DIR / "section2_acf_pacf.png",
)
plt.show()

In [ ]:
# PRINT
print(format_order_selection(order_results))

### Exercise 2: Timeseries Modelling

#### ACF and PACF Results
The discharge series show many significant ACF and PACF lags. This means monthly discharge has memory: the value in one month is still related to previous months. Gisingen_Q and Diepoldsau_Q have quite similar significant-lag patterns, so using the same candidate model orders for Q at both stations is reasonable.

The concentration series also have significant autocorrelation, but the pattern is less clean than for Q. That is expected, because sediment concentration can react strongly to short events, sediment supply, and local river conditions. To keep the comparison simple, we used capped candidate orders: AR(6) and ARMA(6,6). This keeps the models understandable instead of making the order too large just because many lags are significant.

#### Interpretation
Using the same candidate order for both stations is a practical modelling choice, not proof that both rivers work in exactly the same way. The final decision still needs the residual checks in Section 3, because a model order is only useful if it removes most of the remaining autocorrelation.

## Section 3: Timeseries application & evaluation

In [ ]:
# MAIN
# Fit the AR and ARMA candidates, then diagnose residual autocorrelation and normality.
evaluation_results = evaluate_model_collection(
    normalized_series,
    order_results,
    nlags=24,
    alpha=0.05,
)

In [ ]:
# PLOT
fig_section3 = plot_model_diagnostics(
    evaluation_results,
    FIGURE_DIR / "section3_model_diagnostics.png",
)
plt.show()

In [ ]:
# PRINT
print(format_model_evaluation(evaluation_results))

### Exercise 3: Timeseries Application and Evaluation

#### Model Choice
For all four series, ARMA(6,6) gives lower AIC/BIC values than AR(6), so ARMA(6,6) is chosen as the final model in this run. This suggests that adding the moving-average part helps describe the monthly time-series structure better than using AR terms only.

#### Residual Diagnostics
The concentration models have the better residual results. For Gisingen_C and Diepoldsau_C, the ARMA residual Ljung-Box p-values are 0.9576 and 0.7763, so there is no significant remaining autocorrelation at the 5% level. For the two discharge series, the ARMA model improves the fit, but the residuals still fail the Ljung-Box test: p = 2.039e-06 for Gisingen_Q and p = 0.0001079 for Diepoldsau_Q. This means some discharge memory is still left in the residuals.

The residual normality tests reject normality for all four series. We would treat this as an important warning rather than a reason to throw away the whole analysis. River data are often skewed and event-driven, so the models are useful approximations, but they should not be over-interpreted for extreme values.

#### Final Comment
ARMA is more appropriate than AR here because it gives better fit statistics and cleaner residuals, especially for concentration. The selected order is still a compromise: it is simple enough to explain, but the Q residuals show that real discharge behaviour is more complicated than this univariate model can fully capture.

## Section 4: Ill to Rhein relative sediment influence

In [ ]:
# MAIN
sediment_results = run_sediment_influence_analysis(
    monthly_data, review_results, evaluation_results, periods=120, n_paths=10, seed=42
)

# save synth contrib if we have data
contrib_df = sediment_results.get("contrib")
if contrib_df is not None and not contrib_df.empty:
    contrib_df.to_csv(TABLE_DIR / "section4_synthetic_contribution.csv", index=False)


In [ ]:
# PLOT
plot_synthetic_series(
    sediment_results, review_results, FIGURE_DIR / "section4_synthetic_normalized_series.png"
)
plot_sediment_yields(
    sediment_results, FIGURE_DIR / "section4_sediment_yields.png"
)

plt.show()

In [ ]:
# PRINT
print(format_sediment_influence(sediment_results))

for label, result in sediment_results["synth"].items():
    print(f"\n--- Synthetic stats: {label} ---")
    print(result["stats"].round(4))

### Exercise 4: Ill to Rhein Relative Sediment Influence

#### Methodological Context
Stochastic simulation captures long-term statistical properties rather than chronological history. While it cannot replicate specific historical floods, it remains valid for long-term transboundary water planning, baseline variability mapping, and regional yield forecasting.

#### Budget and Yield Summary
* **Observed Baseline:** Historical mean mass rates are 7.642 kg/s (around 2.018 $\times 10^4 \text{ t/month}$) at Gisingen and 189.3 kg/s (around 4.982 $\times 10^5 \text{ t/month}$) at Diepoldsau. 
* **Catchment Contribution:** Over 106 overlapping months, the Ill River contributes an average of **9.26%** of the sediment mass to the downstream Rhein.
* **Synthetic Performance:** Across 10 synthetic paths, the mean relative contribution is tightly preserved at **8.205%**, spanning from **5.711%** (Path 5) to **10.18%** (Paths 1 and 7).

#### Key Limitations
These independent paths underestimate true tail-end sediment risk. Decoupling the physical dependency between discharge ($Q$) and concentration ($C$) distorts peak joint transport ($M = C \cdot Q$). This decoupling produces unphysical negative concentrations that required zero-clipping (**449 times** for Gisingen_C; **781 times** for Diepoldsau_C), proving that independent simulations fail to capture extreme transport events.

## Section 5: Independent variables?

In [ ]:
# MAIN
dependency_results = run_dependency_analysis(monthly_data)

In [ ]:
# PLOT
fig_section5 = plot_dependency_scatter(
    dependency_results,
    FIGURE_DIR / "section5_q_c_dependency.png",
)
plt.show()

In [ ]:
# PRINT
print(format_dependency_results(dependency_results))

### Exercise 5: Q-C Independence Test

#### Evidence Against Independence
Independence between Discharge ($Q$, $m^3/s$) and Concentration ($C$, $g/L$) is rejected ($p \ll 0.05$):
* **Gisingen ($n=192$):** Pearson $r = 0.470$, Spearman $\rho = 0.676$, Kendall $\tau = 0.476$.
* **Diepoldsau ($n=169$):** Pearson $r = 0.584$, Spearman $\rho = 0.738$, Kendall $\tau = 0.538$.

#### Joint Distribution and Mechanics
The plots show a positive, non-linear sediment rating curve ($C = aQ^b$). Higher flows increase erosive capacity, while the widening scatter at peak discharge points to seasonal sediment hysteresis.

#### Shortfalls and Solutions
Decoupled univariate models break the physical link between $Q$ and $C$. Randomly pairing peak floods with baseline concentrations smooths the mass yield curves ($M = C \cdot Q$), underestimating downstream siltation risks.

**Future Fixes:**
1. **Conditional Rating Curves:** Generate $Q$ first, then derive $C$ via $C = aQ^b + \epsilon$.
2. **VAR/VARMA Models:** Simulate both variables simultaneously to keep cross-correlations.
3. **Copulas:** Use tail-dependent copulas to model extreme joint events.